# Day 13 - Validating CSV files

Each task has its own section. Run the notebook from the top with **Run All**.

## Setup

In [ ]:
from pathlib import Path
import csv
import math
import re
from datetime import datetime

INPUT_DIR = Path("olist_data")
BAD_RECORDS_FILE = Path("bad_records.csv")
FILES_TO_VALIDATE = ["olist_orders_dataset.csv", "order_items.csv", "customers.csv"]

print("Input folder:", INPUT_DIR)
print("Bad records file:", BAD_RECORDS_FILE)


## Task 1: required fields for each file

In [ ]:
REQUIRED_FIELDS = {
    "olist_orders_dataset.csv": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
    "order_items.csv": ["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"],
    "customers.csv": ["customer_id", "customer_zip_code_prefix"],
}

for filename, fields in REQUIRED_FIELDS.items():
    print(filename, "->", fields)


## Task 2: validate every row and collect all errors
The checks used here are defined in tasks 4 to 8. The run section at the bottom calls this code and shows the rejected rows with their errors.

In [ ]:
def field_checks() -> dict:
    """Map each column to the checks that apply to it, in the order they run."""
    return {
        "order_id": [check_id],
        "customer_id": [check_id],
        "product_id": [check_id],
        "seller_id": [check_id],
        "customer_zip_code_prefix": [check_zip],
        "order_status": [check_status],
        "order_purchase_timestamp": [check_timestamp],
        "order_delivered_customer_date": [check_timestamp],
        "price": [check_number, check_positive_price],
        "freight_value": [check_number, check_non_negative_freight],
    }


def validate_row(row: dict, filename: str) -> list[str]:
    """Return every problem in one row. An empty list means the row is valid."""
    errors = []
    for col in REQUIRED_FIELDS[filename]:
        if (row.get(col) or "").strip() == "":
            errors.append(f"{col} is blank")
    for col, checks in field_checks().items():
        value = (row.get(col) or "").strip()
        if value == "":
            continue                      # blanks are handled by the required-field check
        for check in checks:
            problem = check(value)
            if problem:
                errors.append(f"{col}: {problem}")
                break                     # stop at the first problem for this column
    return errors


def validate_file(filename: str) -> dict:
    """Read one file, check its columns, check every row, and collect the rejected rows."""
    header, rows = read_input_file(filename)     # task 8
    check_schema(filename, header)               # task 7

    valid_count = 0
    rejected = []
    for line_no, row in enumerate(rows, start=2):    # line 1 is the header
        errors = validate_row(row, filename)
        if errors:
            record = "; ".join(f"{k}={v or ''}" for k, v in row.items() if k is not None)
            rejected.append({
                "file": filename,
                "line": line_no,
                "reason": "; ".join(errors),
                "record": record,
            })
        else:
            valid_count += 1

    return {"file": filename, "total": len(rows), "valid_count": valid_count,
            "rejected": rejected, "status": "OK"}


## Task 3: write rejected rows, with their reasons, to bad_records.csv
The file is written in the run section below.

In [ ]:
def write_bad_records(all_results: list[dict], path: Path = BAD_RECORDS_FILE) -> int:
    """Write every rejected row, with its reason, to a CSV file. Return the number written."""
    rejected = [rec for result in all_results for rec in result["rejected"]]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "line", "reason", "record"])
        writer.writeheader()
        writer.writerows(rejected)
    return len(rejected)


## Task 4: data types - price and freight must be numbers, dates must parse

In [ ]:
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"


def check_number(value: str) -> str | None:
    """Return None if the value is a finite number, otherwise the reason."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if math.isfinite(number) else f"'{value}' is not a finite number"


def check_timestamp(value: str) -> str | None:
    """Return None if the value parses as a date, otherwise the reason."""
    try:
        datetime.strptime(value, DATE_FORMAT)
        return None
    except ValueError:
        return f"'{value}' is not a date in the format YYYY-MM-DD HH:MM:SS"


def show(check, value):
    print(f"{value!r:<26} -> {check(value) or 'OK'}")


print("Numbers:")
show(check_number, "19.18")
show(check_number, "abc")
print("Dates:")
show(check_timestamp, "2018-03-04 12:48:00")
show(check_timestamp, "2018-02-30 10:00:00")
show(check_timestamp, "yesterday")


## Task 5: business rules - price > 0, freight >= 0, status in the allowed list

In [ ]:
ALLOWED_STATUSES = {"approved", "created", "processing", "invoiced",
                    "shipped", "delivered", "unavailable", "canceled"}


def check_positive_price(value: str) -> str | None:
    """Price must be greater than 0."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if number > 0 else "price must be greater than 0"


def check_non_negative_freight(value: str) -> str | None:
    """Freight must be 0 or more."""
    try:
        number = float(value)
    except ValueError:
        return f"'{value}' is not a number"
    return None if number >= 0 else "freight must be 0 or more"


def check_status(value: str) -> str | None:
    """Order status must be one of the allowed statuses."""
    return None if value in ALLOWED_STATUSES else f"status '{value}' is not allowed"


print("Price:")
show(check_positive_price, "19.18")
show(check_positive_price, "0")
print("Freight:")
show(check_non_negative_freight, "-1")
print("Status:")
show(check_status, "delivered")
show(check_status, "lost")


## Task 6: regex - zip prefix is 5 digits, IDs are 32 hex characters

In [ ]:
ZIP_PATTERN = re.compile(r"[0-9]{5}")
ID_PATTERN = re.compile(r"[0-9a-fA-F]{32}")


def check_zip(value: str) -> str | None:
    """Zip prefix must be exactly 5 digits."""
    return None if ZIP_PATTERN.fullmatch(value) else "zip prefix must be 5 digits"


def check_id(value: str) -> str | None:
    """IDs must be exactly 32 hexadecimal characters."""
    return None if ID_PATTERN.fullmatch(value) else "must be 32 hex characters"


print("Zip prefix:")
show(check_zip, "14409")
show(check_zip, "1046")
print("IDs:")
show(check_id, "5d835b501c8e0eb0ce83c72a17cb101a")
show(check_id, "not-a-valid-id")


## Task 7: custom exceptions, and the column check that raises SchemaError

In [ ]:
class SchemaError(Exception):
    """A file is missing required columns."""


class FileFormatError(Exception):
    """A file is empty, has no header row, or is not valid UTF-8."""


def check_schema(filename: str, header: list[str]) -> None:
    """Raise SchemaError if a required column is missing."""
    missing = [col for col in REQUIRED_FIELDS[filename] if col not in header]
    if missing:
        raise SchemaError(f"{filename} is missing required columns: {missing}")


try:
    check_schema("customers.csv", ["customer_id"])
except SchemaError as err:
    print("SchemaError:", err)


## Task 8: handle a missing or empty file without stopping the run

In [ ]:
import tempfile


def read_rows(path: Path) -> tuple[list[str], list[dict]]:
    """Read a CSV file and return (header, rows). Raise FileFormatError if it is empty."""
    try:
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            header = reader.fieldnames
            rows = list(reader)
    except UnicodeDecodeError as err:
        raise FileFormatError(f"{path.name} is not valid UTF-8: {err}") from err
    if not header:
        raise FileFormatError(f"{path.name} is empty or has no header row")
    return header, rows


def read_input_file(filename: str) -> tuple[list[str], list[dict]]:
    """Read a file from INPUT_DIR. Raise FileNotFoundError if it does not exist."""
    path = INPUT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"{filename} not found in {INPUT_DIR}")
    return read_rows(path)


print("Missing file:")
try:
    read_input_file("does_not_exist.csv")
except FileNotFoundError as err:
    print("  Handled:", err)

print("Empty file:")
with tempfile.TemporaryDirectory() as tmp:
    empty = Path(tmp) / "empty.csv"
    empty.write_text("", encoding="utf-8")
    try:
        read_rows(empty)
    except FileFormatError as err:
        print("  Handled:", err)


## Task 9: summary of total, valid and rejected rows

In [ ]:
def print_summary(all_results: list[dict]) -> None:
    """Print total, valid and rejected rows for each file, then the overall totals."""
    print(f"{'File':<30}{'Total':>8}{'Valid':>8}{'Rejected':>10}  Status")
    print("-" * 90)
    for r in all_results:
        print(f"{r['file']:<30}{r['total']:>8}{r['valid_count']:>8}{len(r['rejected']):>10}  {r['status']}")
    total = sum(r["total"] for r in all_results)
    valid = sum(r["valid_count"] for r in all_results)
    rejected = sum(len(r["rejected"]) for r in all_results)
    print("-" * 90)
    print(f"Overall: total={total}, valid={valid}, rejected={rejected}")


## Task 10: fail a whole file when too many rows are rejected (configurable)

In [ ]:
MAX_REJECT_RATE = 0.20   # change this value to set the limit


def apply_reject_threshold(result: dict) -> None:
    """Mark the whole file FAILED if the share of rejected rows is above the limit."""
    if result["total"] == 0:
        return
    rate = len(result["rejected"]) / result["total"]
    if rate > MAX_REJECT_RATE:
        result["status"] = f"FAILED: {rate:.0%} rejected, limit is {MAX_REJECT_RATE:.0%}"


print("Limit:", MAX_REJECT_RATE)
sample_low = {"file": "sample_a.csv", "total": 10, "valid_count": 9, "rejected": [{}] * 1, "status": "OK"}
sample_high = {"file": "sample_b.csv", "total": 10, "valid_count": 6, "rejected": [{}] * 4, "status": "OK"}
apply_reject_threshold(sample_low)
apply_reject_threshold(sample_high)
print(sample_low["file"], "->", sample_low["status"])
print(sample_high["file"], "->", sample_high["status"])


## Run section: validates every file, then shows the summary (task 9) and writes bad_records.csv (task 3)

In [ ]:
all_results = []
for filename in FILES_TO_VALIDATE:
    try:
        result = validate_file(filename)
        apply_reject_threshold(result)                       # task 10
    except FileNotFoundError as err:                         # task 8: missing file is skipped
        result = {"file": filename, "total": 0, "valid_count": 0, "rejected": [],
                  "status": f"SKIPPED: {err}"}
    except (SchemaError, FileFormatError) as err:            # task 7: bad file fails alone
        result = {"file": filename, "total": 0, "valid_count": 0, "rejected": [],
                  "status": f"FAILED: {err}"}
    all_results.append(result)

print_summary(all_results)                                   # task 9

n = write_bad_records(all_results)                           # task 3
print(f"\nWrote {n} rejected rows to {BAD_RECORDS_FILE}")
for line in BAD_RECORDS_FILE.read_text(encoding="utf-8").splitlines()[:5]:
    print(line)
